# Discussion Code — Pandas

**For the instructor:** none of the code cells below have been run. For each one, show the code, read it with the students, and ask **"What do you think will print?"** before running it. Give them a moment to guess, then run the cell to reveal the real answer and discuss why.

These cover indexing (`loc` vs `iloc`), index alignment, missing data, `groupby`, merging, and a few genuine gotchas — several are deliberately tricky. (Verified on pandas 3.0, which has Copy-on-Write always enabled.)

### Question 1
We filter a DataFrame into `subset`, then modify a column on `subset` directly. Does the original `df` change too?

In [63]:
import pandas as pd

df = pd.DataFrame({'A': [1, 2, 3, 4], 'B': [10, 20, 30, 40]})
subset = df[df['A'] > 2]
subset['B'] = 999
print(subset)
print(df)

   A    B
2  3  999
3  4  999
   A   B
0  1  10
1  2  20
2  3  30
3  4  40


C:\Users\Hisham\AppData\Local\Temp\ipykernel_22868\899849226.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  subset['B'] = 999


### Question 2
Trying to access a column that doesn't exist in the DataFrame. What happens?

In [64]:
df = pd.DataFrame({'A': [1, 2, 3], 'B': [4, 5, 6]})
print(df['C'])

KeyError: 'C'

### Question 3
`loc` uses labels, `iloc` uses positions. This DataFrame's index isn't the default 0,1,2... Are `df2.loc[10]` and `df2.iloc[0]` the same thing here?

In [ ]:
df2 = pd.DataFrame({'X': [100, 200, 300]}, index=[10, 20, 30])
print(df2.loc[10])
print(df2.iloc[0])

### Question 4
Adding two Series with only partially overlapping indexes. What happens to the labels that don't match on both sides?

In [65]:
s1 = pd.Series([1, 2, 3])
s2 = pd.Series([10, 20, 30])
print(s1 + s2)

0    11
1    22
2    33
dtype: int64


In [66]:
s1 = pd.Series([1, 2, 3], index=['a', 'b', 'c'])
s2 = pd.Series([10, 20, 30], index=['b', 'c', 'd'])
print(s1 + s2)

a     NaN
b    12.0
c    23.0
d     NaN
dtype: float64


### Question 5
A Series with a missing value in it. Do `.sum()` and `.mean()` count the missing value as zero, skip it, or fail?

In [67]:
s = pd.Series([1, 2, None, 4])
print(s.sum())
print(s.mean())

7.0
2.3333333333333335


In [ ]:
s = pd.Series([1, 2, None, "sd",4])
print(s.sum())
print(s.mean())

TypeError: unsupported operand type(s) for +: 'int' and 'str'

### Question 6
`value_counts()` with `normalize=True`. What does it return instead of raw counts?

In [70]:
df3 = pd.DataFrame({'Dept': ['HR', 'IT', 'HR', 'IT', 'IT']})
print(df3['Dept'].value_counts())

Dept
IT    3
HR    2
Name: count, dtype: int64


In [71]:
df3 = pd.DataFrame({'Dept': ['HR', 'IT', 'HR', 'IT', 'IT']})
print(df3['Dept'].value_counts(normalize=True))

Dept
IT    0.6
HR    0.4
Name: proportion, dtype: float64


### Question 7
`duplicated()` on a DataFrame with one repeated row. Which row(s) get flagged as `True`?

In [73]:
df4 = pd.DataFrame({'A': [1, 2, 2, 3], 
                    'B': [5, 6, 6, 7]})
df4

,A,B
0,1,5
1,2,6
2,2,6
3,3,7


In [76]:
print(df4.duplicated())


0    False
1    False
2     True
3    False
dtype: bool


In [ ]:
df4 = pd.DataFrame({'A': [1, 2, 2, 3], 
                    'B': [5, 6, 6, 7]})
df4['A'].duplicated().any() #at least one true ?

True

In [78]:

df1 = df.columns[df.T.duplicated()]
print(df.T.duplicated()) 

A    False
B    False
dtype: bool


### Question 8
A column of numeric-looking strings. What's its dtype before and after `.astype(int)`, and can we `.sum()` it afterward?

In [79]:
df5 = pd.DataFrame({'A': ['1', '2', '3']})
print(df5['A'].dtype)
df5['A'] = df5['A'].astype(int)
print(df5['A'].dtype)
print(df5['A'].sum())

object
int32
6


### Question 9
`groupby()` combined with `.agg()` and a *list* of functions instead of just one. What's the shape of the result?

In [80]:
df6 = pd.DataFrame({'Dept': ['HR', 'IT', 'HR', 'IT'], 'Salary': [50000, 60000, 55000, 70000]})
print(df6.groupby('Dept')['Salary'].agg(['mean', 'max', 'min']))

         mean    max    min
Dept                       
HR    52500.0  55000  50000
IT    65000.0  70000  60000


### Question 10
`apply()` with `axis=1`, using a lambda that reads two columns from the same row. What does the new column contain?

In [81]:
df7 = pd.DataFrame({'A': [1, 2, 3], 'B': [4, 5, 6]})
df7['Sum'] = df7.apply(lambda row: row['A'] + row['B'], axis=1)
print(df7)

   A  B  Sum
0  1  4    5
1  2  5    7
2  3  6    9


### Question 11
Combining two conditions with `&`. Which single row satisfies both at once?

In [82]:
df8 = pd.DataFrame({'Age': [25, 35, 45, 20], 'Dept': ['HR', 'IT', 'HR', 'IT']})
result = df8[(df8['Age'] > 30) & (df8['Dept'] == 'IT')]
print(result)

   Age Dept
1   35   IT


### Question 12
`sort_values()` with a *list* of two column names. Which column gets sorted first?

In [83]:
df9 = pd.DataFrame({'Dept': ['IT', 'HR', 'IT', 'HR'], 'Salary': [40000, 50000, 60000, 55000]})
print(df9.sort_values(by=['Dept', 'Salary']))

  Dept  Salary
1   HR   50000
3   HR   55000
0   IT   40000
2   IT   60000


In [84]:
df9 = pd.DataFrame({'Dept': ['IT', 'HR', 'IT', 'HR'], 'Salary': [40000, 50000, 60000, 55000]})
print(df9.sort_values(by=['Salary', 'Dept']))

  Dept  Salary
0   IT   40000
1   HR   50000
3   HR   55000
2   IT   60000


In [85]:
df9 = pd.DataFrame({'Dept': ['IT', 'HR', 'IT', 'HR'], 'Salary': [40000, 50000, 60000, 55000]})
print(df9.sort_values(by=['Salary', 'Dept'],ascending=False))

  Dept  Salary
2   IT   60000
3   HR   55000
1   HR   50000
0   IT   40000


### Question 13
Renaming a column with a dictionary. What does the result look like?

In [86]:
df10 = pd.DataFrame({'old_name': [1, 2, 3]})
df10 = df10.rename(columns={'old_name': 'new_name'})
print(df10)

   new_name
0         1
1         2
2         3


### Question 14
After filtering, the row index keeps its *original* numbers with gaps. What does `reset_index(drop=True)` do about that?

In [87]:
df11 = pd.DataFrame({'A': [1, 2, 3, 4, 5]})
filtered = df11[df11['A'] > 2]
print(filtered)
print(filtered.reset_index(drop=True))


   A
2  3
3  4
4  5
   A
0  3
1  4
2  5


In [88]:
df11 = pd.DataFrame({'A': [1, 2, 3, 4, 5]})
filtered = df11[df11['A'] > 2]
print(filtered)
print(filtered.reset_index())

   A
2  3
3  4
4  5
   index  A
0      2  3
1      3  4
2      4  5


### Question 15
Merging two DataFrames on a shared `id` column with the default merge type. What happens to `id=3` (only in the first) and `id=4` (only in the second)?

In [89]:
df_a = pd.DataFrame({'id': [1, 2, 3], 'name': ['Ali', 'Sara', 'Omar']})
df_b = pd.DataFrame({'id': [1, 2, 4], 'score': [90, 85, 70]})
merged = pd.merge(df_a, df_b)
print(merged)

   id  name  score
0   1   Ali     90
1   2  Sara     85


In [90]:
df_a = pd.DataFrame({
    'id': [1, 2, 3],
    'name': ['Ali', 'Sara', 'Omar'],
    'city': ['Cairo', 'Giza', 'Alex']
})

df_b = pd.DataFrame({
    'id': [1, 2, 4],
    'city': ['Cairo', 'Giza', 'Cairo'],
    'score': [90, 85, 70]
})

merged = pd.merge(df_a, df_b)

print(merged)

   id  name   city  score
0   1   Ali  Cairo     90
1   2  Sara   Giza     85


### Question 16
Same merge, but this time with `how='left'`. Does Omar (id=3) show up now, and what value does he get for `score`?

In [ ]:
merged_left = pd.merge(df_a, df_b, on='id', how='left')
print(merged_left)

   id  name city_x city_y  score
0   1   Ali  Cairo  Cairo   90.0
1   2  Sara   Giza   Giza   85.0
2   3  Omar   Alex    NaN    NaN


In [92]:
merged_left = pd.merge(df_a, df_b, on='id', how='right')
print(merged_left)

   id  name city_x city_y  score
0   1   Ali  Cairo  Cairo     90
1   2  Sara   Giza   Giza     85
2   4   NaN    NaN  Cairo     70


### Question 17
`.unique()` vs `.nunique()` on the same column. What's the difference in what they return?

In [93]:
df12 = pd.DataFrame({'Dept': ['HR', 'IT', 'HR', 'Finance', 'IT']})
print(df12['Dept'].unique())
print(df12['Dept'].nunique())

['HR' 'IT' 'Finance']
3


### Question 18
Using `.str.upper()` on a text column. Does this modify the column in place, or do we need to assign it somewhere?

In [ ]:

df13 = pd.DataFrame({'Name': ['ali', 'sara', 'omar'],'age':[25,26,27]})

df16['Name'] = df13['Name'].str.upper()
df16['age'] = df13['age']
print(df16)

0                                                     ALI
1                                                    SARA
2                                                    OMAR
age       0    25
1    26
2    27
Name: age, dtype: int64
Name    0     ALI
1    SARA
2    OMAR
Name: Name, dtyp...
Name: Name, dtype: object


### Question 19
`.ffill()` (forward-fill) on a Series with two consecutive missing values. What do they get filled with?

In [100]:
s3 = pd.Series([1, None, None, 4])
print(s3.ffill())

0    1.0
1    1.0
2    1.0
3    4.0
dtype: float64


### Question 20
Comparing two DataFrames with identical content using `==` versus `.equals()`. Do they give the same *type* of result?

In [103]:
df14 = pd.DataFrame({'A': [1, 2, 3]})
df15 = pd.DataFrame({'A': [1, 2, 3]})
print(df14 == df15)
print(df14.equals(df15))

      A
0  True
1  True
2  True
True
